# Lab05.3 – Traffic Sign Recognition

**Objective:** combine colour, geometry and one interior feature for a simple rule-based classifier.

This is a controlled educational dataset, not a road-safe or general traffic-sign recognition system. The classifier distinguishes STOP, NO ENTRY, WARNING, PARKING and MANDATORY DIRECTION; the speed limit is only flagged as a candidate, not read by OCR.

**Step 1 — Preparation.** Import the libraries, specify the image folder and define the image display function.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

IMAGE_DIR = Path('images')
def show_bgr(img, title='Image'):
    plt.figure(figsize=(11, 6))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()

print('OpenCV:', cv2.__version__)

**Step 2 — HSV masks.** Define red/blue segmentation. The resulting masks will be displayed in the next step.

In [ ]:
def color_masks(bgr):
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    # OpenCV hue uses 0..179. Red wraps around zero.
    red1 = cv2.inRange(hsv, (0, 90, 70), (12, 255, 255))
    red2 = cv2.inRange(hsv, (168, 90, 70), (179, 255, 255))
    red_mask = cv2.bitwise_or(red1, red2)
    blue_mask = cv2.inRange(hsv, (95, 90, 60), (130, 255, 255))
    kernel = np.ones((3, 3), np.uint8)
    red_mask = cv2.morphologyEx(red_mask, cv2.MORPH_OPEN, kernel)
    blue_mask = cv2.morphologyEx(blue_mask, cv2.MORPH_OPEN, kernel)
    return {'red': red_mask, 'blue': blue_mask}

**Visual checkpoint — Colour masks.** The next cell reads the input and displays red and blue masks. If the segmentation is not correct, later contour and shape detection cannot succeed.

In [ ]:
img = cv2.imread(str(IMAGE_DIR / 'traffic_signs.png'))
assert img is not None, 'Missing images/traffic_signs.png'
masks = color_masks(img)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); axes[0].set_title('Input')
axes[1].imshow(masks['red'], cmap='gray', vmin=0, vmax=255); axes[1].set_title('Red mask')
axes[2].imshow(masks['blue'], cmap='gray', vmin=0, vmax=255); axes[2].set_title('Blue mask')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

**Step 4 — Candidate contours.** Define contour extraction and filter tiny components that are unlikely to be signs.

In [ ]:
def find_contours(mask):
    result = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return result[-2]  # works with OpenCV 3 and OpenCV 4

def candidates(bgr, min_area=400):
    out = []
    for color, mask in color_masks(bgr).items():
        for cnt in find_contours(mask):
            area = cv2.contourArea(cnt)
            if area < min_area: continue
            x, y, w, h = cv2.boundingRect(cnt)
            if w < 18 or h < 18: continue
            out.append({'color': color, 'contour': cnt,
                        'bbox': (x, y, w, h), 'area': area})
    return sorted(out, key=lambda c: (c['bbox'][1], c['bbox'][0]))

**Visual checkpoint — Candidate regions.** Highlight all accepted colour contours and their bounding boxes. These are only potential signs, not yet recognized shapes.

In [ ]:
preview = img.copy()
found = candidates(img)
for item in found:
    x, y, w, h = item['bbox']
    cv2.rectangle(preview, (x, y), (x + w, y + h), (0, 180, 0), 2)
show_bgr(preview, 'Colour-based sign candidates')
print('Candidates:', len(found))

**Step 6 — Shape analysis.** Define a polygon-based shape classifier: triangle, quadrilateral, circle or octagon.

In [ ]:
def shape_of(contour):
    perimeter = cv2.arcLength(contour, True)
    if perimeter == 0: return 'unknown', 0
    approx = cv2.approxPolyDP(contour, 0.035 * perimeter, True)
    vertices = len(approx)
    if vertices == 3: return 'triangle', vertices
    if vertices == 4: return 'quadrilateral', vertices
    (_, _), radius = cv2.minEnclosingCircle(contour)
    fill_ratio = cv2.contourArea(contour) / (np.pi * radius * radius)
    if fill_ratio > 0.96: return 'circle', vertices
    if 7 <= vertices <= 10: return 'octagon', vertices
    return 'other', vertices

**Visual checkpoint — Shape labels.** Show the shape computed for every candidate before combining it with colour to recognize signs.

In [ ]:
preview = img.copy()
for item in found:
    shape, vertices = shape_of(item['contour'])
    x, y, w, h = item['bbox']
    cv2.rectangle(preview, (x, y), (x+w, y+h), (0, 180, 0), 2)
    cv2.putText(preview, shape, (x, max(18, y-6)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)
show_bgr(preview, 'Candidate shapes before sign recognition')

**Step 7 — Traffic-sign decision rules.** Define how colour and shape map to candidate sign classes. For red circular signs, inspect the white horizontal centre stripe to distinguish NO ENTRY. This cell defines a function and has no image output.

In [ ]:
def classify(candidate, bgr):
    color = candidate['color']
    shape, _ = shape_of(candidate['contour'])
    if color == 'blue':
        if shape == 'quadrilateral': return 'PARKING'
        if shape == 'circle': return 'MANDATORY DIRECTION'
    if color == 'red':
        if shape == 'triangle': return 'WARNING'
        if shape == 'octagon': return 'STOP'
        if shape == 'circle':
            x, y, w, h = candidate['bbox']
            roi = bgr[y:y+h, x:x+w]
            hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
            # Detect a large central horizontal white bar (NO ENTRY).
            center = hsv[int(h*.46):int(h*.54), int(w*.30):int(w*.70)]
            if center.size:
                white = cv2.inRange(center, (0, 0, 180), (179, 70, 255))
                if np.mean(white > 0) > 0.80: return 'NO ENTRY'
            return 'SPEED LIMIT (candidate)'
    return 'UNCLASSIFIED'

**Step 9 — Recognition on the reference scene.** Apply all defined functions to `traffic_signs.png`, draw the class labels and print the detected classes.

In [ ]:
def recognize(filename):
    img = cv2.imread(str(IMAGE_DIR / filename))
    assert img is not None, 'Image missing: ' + filename
    ann = img.copy()
    result = []
    for c in candidates(img):
        label = classify(c, img)
        x, y, w, h = c['bbox']
        cv2.rectangle(ann, (x,y), (x+w,y+h), (0, 135, 0), 2)
        cv2.putText(ann, label, (x, max(20,y-8)), cv2.FONT_HERSHEY_SIMPLEX, .52, (0,0,0), 2)
        result.append(label)
    show_bgr(ann, filename)
    return result

results = recognize('traffic_signs.png')
print('Detected:', results)

## 1. Unseen combinations of the same synthetic signs
Run the classifier on three different scenes.

**Step 10 — Generalization check.** Run the same recognition algorithm on three additional synthetic scenes. Observe any missing or incorrect detections.

In [ ]:
for filename in ['test_01.png', 'test_02.png', 'test_03.png']:
    print(filename, recognize(filename))

## 2. Student investigation
1. Explain how colour reduces the number of possible sign classes.
2. Which two red circular signs need additional interior analysis?
3. Why is `SPEED LIMIT (candidate)` not full recognition of the number?
4. Test a photograph of a real traffic sign. Document at least two failure modes.

## Conclusion
The next laboratories can accelerate thresholding or mask generation in FPGA logic, then compare results and PS–PL transfer overhead against OpenCV.